In [12]:
import pandas as pd
import ROOT
import numpy as np
from flarefly.data_handler import DataHandler
from flarefly.fitter import F2MassFitter

In [13]:
dataset = pd.read_parquet('Dataset.parquet')
list(dataset)

['fCandidateSelFlag',
 'fParticleFlag',
 'fOriginMcRec',
 'fPt',
 'fM',
 'fCt',
 'fDecayLength',
 'fDecayLengthNormalised',
 'bkg_score',
 'prompt_score',
 'non_prompt_score']

In [14]:
# 1. Create the ROOT histogram (name, title;xaxis;yaxis, bins, min, max)
h_fM = ROOT.TH1F("h_fM", "Invariant Mass; Invariant Mass; Reconstructed Events", 100, 2.36, 2.59)

# 2. Extract the Pandas column as a flat NumPy array of doubles
data_array = dataset['fM'].to_numpy(dtype=np.float64)
weights = np.ones(len(data_array), dtype=np.float64)

# 3. Fill the histogram efficiently
h_fM.FillN(len(data_array), data_array, weights)

# 4. Draw it on a canvas
canvas = ROOT.TCanvas("c1", "c1", 800, 600)
h_fM.Draw()
canvas.Draw()

Warning in <TROOT::Append>: Replacing existing TH1: h_fM (Potential memory leak).
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1


In [19]:
h_pT = ROOT.TH1F("h_pT", "pT; pT; Reconstructed Events", 100, 1, 12)

# 2. Extract the Pandas column as a flat NumPy array of doubles
data_array = dataset['fPt'].to_numpy(dtype=np.float64)
weights = np.ones(len(data_array), dtype=np.float64)

# 3. Fill the histogram efficiently
h_pT.FillN(len(data_array), data_array, weights)

# 4. Draw it on a canvas
canvas = ROOT.TCanvas("c1", "c1", 800, 600)
h_pT.Draw()
canvas.Draw()

Warning in <TROOT::Append>: Replacing existing TH1: h_pT (Potential memory leak).
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1


In [ ]:
data = DataHandler(data = dataset, var_name='fM', limits=[2.36, 2.59])
fitter = F2MassFitter(
    data_handler = data, 
    name_signal_pdf = 'doublegaus', 
    name_background_pdf = 'powlaw',
    label_bkg_pdf = 'Combinatorial background',
    name="powlaw_doublegaus"
    )

fitter.set_signal_initpar(double_gauss_id, "mu", 2.468)
fitter.set_signal_initpar(double_gauss_id, "frac1", 0.8)
fitter.set_signal_initpar(double_gauss_id, "sigma1", 0.008)
fitter.set_signal_initpar(double_gauss_id, "sigma2", 0.022)

fitter.mass_zfit()
dataset['sWeight']= fitter.get_sweights()['signal']